# Palavra_Status_Lista_CCTV_ExB_v30: "online" / "offline" escrito na lista de câmeras (pedido do Ricardo)

Na lista de câmeras, a linha do bairro passa a mostrar também a palavra do status, colorida:
- **Centro · online** (verde)
- **Centro · offline** (vermelho)

A bolinha em cima do ícone continua. O placar no topo continua.
A palavra vem dos campos `txt_online` / `txt_offline`, que o **Atualiza_Status_Cameras_CCTV_v4** cria e preenche.

**Ordem:** 1) rode o Atualiza_Status v4 → 2) feche a aba do construtor → 3) **Run All** neste →
4) Ctrl+F5 → mudança pequena → Salvar → Publicar → zip `cctv_v15.zip`.

In [ ]:
# =====================================================================
# CÉLULA 1 de 2 - ACHAR O APP E FAZER BACKUP
# =====================================================================
import os, re, json, glob, shutil, copy, uuid, datetime, builtins, warnings
from urllib.parse import quote
warnings.filterwarnings("ignore")
NOME = "Palavra_Status_Lista_CCTV_ExB_v30"
try:
    import arcpy
    _RAIZ = arcpy.mp.ArcGISProject("CURRENT").homeFolder
except Exception:
    _RAIZ = os.path.join(os.path.expanduser("~"), "Documents")
_SAIDA = next((os.path.join(_RAIZ, n) for n in ("Saidas", "Saida") if os.path.isdir(os.path.join(_RAIZ, n))), os.path.join(_RAIZ, "Saidas"))
os.makedirs(_SAIDA, exist_ok=True)
LOG_TXT = os.path.join(_SAIDA, NOME + ".txt"); _LOG = open(LOG_TXT, "w", encoding="utf-8")
_p = builtins.print
def print(*a, **k):
    _p(*a, **k); k.pop("file", None); _p(*a, **k, file=_LOG); _LOG.flush()
print("▶ Célula 1 iniciada...")
RAIZ = r"C:\ExB_1.19\server\public\apps"
APP = None
for d in sorted(glob.glob(os.path.join(RAIZ, "*")), key=os.path.getmtime, reverse=True):
    cfg = os.path.join(d, "resources", "config", "config.json")
    if os.path.isfile(cfg) and "4e15fb40c0e346aba97b0dccd0032048" in open(cfg, encoding="utf-8").read():
        APP = d; break
if not APP: raise RuntimeError("PAREI: não achei o app de CCTV.")
CFG_PATH = os.path.join(APP, "resources", "config", "config.json")
with open(CFG_PATH, encoding="utf-8") as f: C = json.load(f)
W = C["widgets"]
ALVO = "widget_392"   # texto do bairro na lista de câmeras
if ALVO not in W or "bairro" not in json.dumps(W[ALVO].get("useDataSources")):
    raise RuntimeError("PAREI: não achei o texto do bairro da lista (widget_392).")
if "TXT_ONLINE" in W[ALVO]["config"]["text"]:
    raise RuntimeError("PAREI: a palavra online/offline já está na lista (o v30 já foi rodado).")
BKP = os.path.join(os.path.expanduser("~"), "Documents", "Backup_ExB", f"cctv_antes_v30_{datetime.datetime.now():%Y%m%d_%H%M%S}")
shutil.copytree(APP, BKP)
print(f"   App: {APP} · Backup: {BKP}")
print("✔ Célula 1 concluída com sucesso.")


In [ ]:
# =====================================================================
# CÉLULA 2 de 2 - PALAVRA DO STATUS NA LINHA DO BAIRRO
# =====================================================================
print("▶ Célula 2 iniciada...")
DS = W[ALVO]["useDataSources"][0]["dataSourceId"]
def _uid():
    h = uuid.uuid4().hex; return f"{h[:8]}_{h[8:12]}_{h[12:16]}_{h[16:20]}_{h[20:32]}"
def _exp(campo, rot):
    e = {"name": "{%s}" % rot, "parts": [{"type": "FIELD", "exp": "{%s}" % rot, "dataSourceId": DS,
                                          "jimuFieldName": campo, "isFromRepeatedDataSourceContext": True}]}
    return (f'<exp data-uniqueid="{_uid()}" data-dsid="{DS}" data-expression="{quote(json.dumps(e, separators=(",", ":")), safe="")}">'
            f'\ufeff<span contenteditable="false">\ufeff\ufeff{{{rot}}}\ufeff\ufeff</span>\ufeff</exp>')
W[ALVO]["config"]["text"] = (
    '<p><span style="font-size:12px;color:#6b7690;">' + _exp("bairro", "BAIRRO") + '&nbsp;·&nbsp;</span>'
    '<span style="font-size:12px;font-weight:600;color:#1f9d55;">' + _exp("txt_online", "TXT_ONLINE") + '</span>'
    '<span style="font-size:12px;font-weight:600;color:#d64545;">' + _exp("txt_offline", "TXT_OFFLINE") + '</span></p>')
for u in W[ALVO]["useDataSources"]:
    u["fields"] = sorted(set(u.get("fields", [])) | {"bairro", "txt_online", "txt_offline"})
W[ALVO]["useDataSourcesEnabled"] = True
with open(CFG_PATH, "w", encoding="utf-8") as f: json.dump(C, f, ensure_ascii=False, indent=2)
print(f"   ✔ Lista: linha do bairro agora mostra 'Bairro · online' (verde) ou 'Bairro · offline' (vermelho)")
print("   App gravado.")
print("")
print("✔ Célula 2 concluída com sucesso.")
print("Próximo passo: Ctrl+F5 no construtor → mudança pequena → Salvar → Publicar → zip cctv_v15.zip")
print("Links:")
print("   App (rascunho): https://localhost:3001/experience/2/?draft=true")
print("   App (publicado): https://localhost:3001/experience/2/")
print(f"   Desfazer: copie de volta {BKP} para {APP}")
print(f"   Registro: {LOG_TXT}")
